# Week 2: Perceptron, Backpropagation, and the Training Loop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rrfhwn/neural-architectures-and-representation-learning-course/blob/main/weeks/02/Week_02_Perceptron_Backprop_TrainingLoop.ipynb)

## Learning goals

- Formalise the **single neuron**: $z = w^\top x + b$, activation, and the perceptron update rule.
- Understand **why we move from a step function to a differentiable loss** (logistic regression intuition).
- Master the **training loop**: batch, epoch, forward pass, loss, backward pass, optimizer step—with clear pseudocode.
- Implement a **perceptron from scratch** on 2D toy data and visualise the decision boundary.
- See **XOR failure** of a linear classifier and how a **tiny MLP** solves it.
- Run a **minimal MNIST training loop** (1–2 epochs) to connect theory to practice.

### Quick recap (from Week 1)

- Linear decision boundary: $f(x) = w^\top x + b = 0$ is a hyperplane.
- Perceptron: one neuron, sign output; converges only when data are linearly separable.
- XOR: no single line can separate the four points; linear models fail.
- Hidden layers + non-linearity: the network learns a feature map so the last layer can separate classes linearly in the new space.
- Universal approximation: wide enough MLPs can approximate continuous functions.

---
## Environment

### Local (uv)

From the repo root:

```bash
uv sync
uv run jupyter notebook weeks/02/Week_02_Perceptron_Backprop_TrainingLoop.ipynb
```

Dependencies used in this notebook: `torch`, `torchvision`, `matplotlib`, `numpy`. No other packages required.

### Colab

1. Open the notebook in Colab via the badge link above.
2. Runtime → Run all. PyTorch and matplotlib are pre-installed; no extra setup needed.
3. Optional: Runtime → Change runtime type → GPU if you want to use CUDA for the MNIST section (the notebook runs fine on CPU with 1–2 epochs).

---
## 1. Foundations: Single Neuron and Perceptron

### Single neuron: $w^\top x + b$ and activation

A **single neuron** takes an input vector $x \in \mathbb{R}^d$, a weight vector $w \in \mathbb{R}^d$, and a scalar bias $b$. It computes:

$$z = w^\top x + b$$

Then an **activation function** $\sigma$ is applied: $a = \sigma(z)$. The output $a$ can be a real number (e.g. for regression or logits) or a binary decision (e.g. $\operatorname{sign}(z)$ for the perceptron).

**Intuition:** $z$ is a weighted sum of the inputs plus a constant offset. The sign of $z$ tells you which side of the hyperplane $w^\top x + b = 0$ the point lies on; the magnitude of $z$ is a measure of confidence (distance from the boundary).

### Perceptron: definition and update rule

The **perceptron** is a binary classifier: output $\hat{y} = \operatorname{sign}(z) \in \{-1, +1\}$ (or equivalently $\{0, 1\}$ depending on convention).

**Update rule (perceptron algorithm):** For each training example $(x, y)$ with true label $y \in \{-1,+1\}$:

1. Compute $\hat{y} = \operatorname{sign}(w^\top x + b)$.
2. If $\hat{y} \neq y$, update:
   - $w \leftarrow w + \eta \, y \, x$
   - $b \leftarrow b + \eta \, y$

where $\eta$ is the learning rate. So we nudge the boundary toward the misclassified point. The perceptron is guaranteed to converge in finitely many steps **only when the data are linearly separable**.

**Perceptron vs. gradient descent:** The perceptron update is a *reactive correction*—it fires only on misclassified points and has no notion of *how wrong* the prediction is. Gradient descent, by contrast, optimises a smooth, differentiable objective and adjusts parameters proportionally to the error magnitude, making it applicable even when classes overlap and enabling learning in deep, multi-layer networks.

### Step function vs differentiable loss (logistic regression intuition)

The perceptron uses a **step function** (sign): output is either $+1$ or $-1$. The "loss" is effectively "wrong or right" with no gradient—we only know *that* we were wrong, not *how much*. The update rule above is a fixed heuristic (move the boundary toward the mistake), not gradient descent.

In **logistic regression**, we use a **differentiable** activation (sigmoid) and a proper loss (e.g. cross-entropy / log loss). Then we can compute $\frac{\partial \mathcal{L}}{\partial w}$ and $\frac{\partial \mathcal{L}}{\partial b}$ and take gradient steps. Benefits:

- **Smooth updates:** small changes in $w$ lead to small changes in the loss; we can follow the gradient.
- **Works with non-separable data:** we minimise average loss instead of waiting for perfect separation.
- **Extends to deep networks:** backpropagation requires differentiable operations end-to-end.

So: step function → simple rule, no gradients, only for separable data. Differentiable loss → gradient descent, works more generally, and is what we use in practice for neural nets.

### Training loop: batch, epoch, forward, backward, step

In modern deep learning we train on **batches** of data and repeat over **epochs**.

- **Batch:** a subset of the training set (e.g. 64 samples). We compute the loss on this subset and take one optimizer step.
- **Epoch:** one full pass through the entire training set (typically in shuffled batches).
- **Forward pass:** given a batch $(X, y)$, compute model output $\hat{y} = f(X; \theta)$ and loss $\mathcal{L}(\hat{y}, y)$.
- **Backward pass:** compute gradients $\frac{\partial \mathcal{L}}{\partial \theta}$ (e.g. via backpropagation).
- **Optimizer step:** update parameters $\theta \leftarrow \theta - \eta \, \nabla_\theta \mathcal{L}$ (or variant like Adam).

**Intuition:** Think of the loss surface as a hilly landscape: each gradient step nudges all parameters a small amount downhill, and with enough steps the model settles into a low-loss valley. Batching means we get a noisy but cheap estimate of the slope at each step instead of computing it over the entire dataset.

**Pseudocode (one training step per batch):**

```
for epoch in 1, 2, ...:
    shuffle(training_data)
    for batch (X, y) in batches(training_data, batch_size):
        # Forward
        logits = model(X)
        loss = criterion(logits, y)
        # Backward
        optimizer.zero_grad()   # clear old gradients
        loss.backward()        # compute new gradients
        optimizer.step()       # update parameters
```

> **Mental model — one training step**
>
> | Stage | What happens |
> |---|---|
> | **Forward pass** | input flows through the network → prediction $\hat{y}$ |
> | **Loss** | compare $\hat{y}$ to ground truth → scalar error signal $\mathcal{L}$ |
> | **Backward pass** | $\mathcal{L}$ propagated back through the graph → gradient $\nabla_\theta \mathcal{L}$ for every parameter |
> | **Optimizer step** | each parameter updated in the direction that reduces the loss → $\theta \leftarrow \theta - \eta\,\nabla_\theta\mathcal{L}$ |

This four-step cycle is repeated for every batch across many epochs.

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────────────────
seed = 12
# np.random.seed(seed) is called in the imports cell below, once NumPy is loaded.
# TODO: Change `seed` to a different value (e.g. 0, 7, 123) and re-run the
#       notebook. Observe how the decision boundary varies across seeds.

#### Pause & Reflect

- Why do we call `optimizer.zero_grad()` before `loss.backward()`? What would happen if we forgot it?
- How does the batch size affect the number of parameter updates per epoch? If you have 60,000 training samples and batch size 64, how many steps per epoch?
- The perceptron update rule does not use gradients. How would you express "move the boundary toward the misclassified point" in terms of a (non-differentiable) "loss"?

---
## 2. Practical 1: Perceptron from scratch on 2D toy data (two-phase experiment)

We run a **two-phase** experiment to see how a perceptron behaves when **new data arrives after initial convergence**. This simulates **distribution shift** or **streaming data**: the model was trained on one dataset, then more data (from the same or a shifted distribution) appears. That motivates why we care about **epochs** (multiple passes), **batching** (how we combine updates), and **training dynamics** (continuing from current weights vs re-initialising). Code here uses only NumPy and Matplotlib; CPU-fast and minimal.

- **Phase A:** Two Gaussian clusters (Class A and Class B), linearly separable. Train the perceptron **online** (sample-by-sample) until convergence or max epochs. Plot data and decision boundary.
- **Phase B:** New data arrives—a third Gaussian cluster belonging to Class B, placed so that the optimal boundary changes. **Continue training from the previous weights** (do not reinitialise). Plot the updated boundary and see how it shifts to include the new cluster.

### Imports (NumPy, Matplotlib; torch only needed later for MLP/MNIST)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

np.random.seed(seed)  # seed defined in the reproducibility cell above

### Phase A: Generate two clusters (Class A and Class B), linearly separable

Each cluster is a Gaussian around one centroid. We map Class A / Class B to numeric labels internally (e.g. +1 and −1) for the perceptron.

In [ ]:
# Fixed axis limits for all toy plots
XLIM = (-10, 10)
YLIM = (-10, 10)

# Fixed centroids and cluster size
MU1 = np.array([-6.0, -2.0])  # Class A
MU2 = np.array([6.0, -2.0])   # Class B (Phase A)
MU3 = np.array([-4.0, 4.0])    # Class B new cluster (Phase B)
N_PER_CLUSTER = 60
CLUSTER_STD = 0.9

def make_two_clusters(n_per_cluster=N_PER_CLUSTER, std=CLUSTER_STD):
    """Two Gaussian clusters in 2D, linearly separable. Class A = +1, Class B = -1.
    Fixed centroids: Class A at μ1=(-6,-2), Class B at μ2=(6,-2).
    Uses the global np.random state (call np.random.seed before invoking).
    """
    c_a = np.random.randn(n_per_cluster, 2) * std + MU1
    c_b = np.random.randn(n_per_cluster, 2) * std + MU2
    X = np.vstack([c_a, c_b])
    y = np.hstack([np.ones(n_per_cluster), -np.ones(n_per_cluster)])
    return X.astype(np.float32), y.astype(np.float32)

In [ ]:
X, y = make_two_clusters()
n_a = np.sum(y == 1)   # number of Class A points (= N_PER_CLUSTER)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(X[y == 1, 0],  X[y == 1, 1],  c='tab:blue',   marker='o', alpha=0.7, label='Class A (μ1)')
ax.scatter(X[y == -1, 0], X[y == -1, 1], c='tab:orange',  marker='s', alpha=0.7, label='Class B (μ2)')
# Centroids
ax.scatter(*MU1, c='k', marker='x', s=120, zorder=5, linewidths=2)
ax.scatter(*MU2, c='k', marker='x', s=120, zorder=5, linewidths=2)
ax.annotate('μ1', MU1, textcoords='offset points', xytext=(6, 4), fontsize=10)
ax.annotate('μ2', MU2, textcoords='offset points', xytext=(6, 4), fontsize=10)
ax.set_xlim(XLIM)
ax.set_ylim(YLIM)
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.legend()
ax.set_title('Phase A data: Class A (μ1) and Class B (μ2) — linearly separable')
plt.tight_layout()
plt.show()

### TODO 1

Change `n_per_cluster` to 100 and re-run the cell above. How does the decision boundary you get later (after training) compare when you use more points?

### Phase A: Train perceptron online (sample-by-sample) until convergence or max epochs

Classic perceptron: one update per sample when misclassified. We run over epochs until no mistakes or `max_epochs` is reached.

In [ ]:
def perceptron_fit(X, y, lr=0.1, max_epochs=100):
    """X: (n, 2), y: (n,) with values in {-1, +1}."""
    n, d = X.shape
    w = np.zeros(d, dtype=np.float32)
    b = 0.0
    for epoch in range(max_epochs):
        mistakes = 0
        for i in range(n):
            z = np.dot(w, X[i]) + b
            pred = 1 if z >= 0 else -1
            if pred != y[i]:
                mistakes += 1
                w = w + lr * y[i] * X[i]
                b = b + lr * y[i]
        if mistakes == 0:
            print(f"Converged at epoch {epoch + 1}")
            break
    return w, b

In [ ]:
w, b = perceptron_fit(X, y)
print(f"w = {w}, b = {b}")

In [ ]:
def add_third_cluster(X, y, n_extra=N_PER_CLUSTER, std=CLUSTER_STD):
    """Append μ3-cluster for Class B (label −1).  Uses global np.random state."""
    extra = np.random.randn(n_extra, 2) * std + MU3
    X_new = np.vstack([X, extra]).astype(np.float32)
    y_new = np.hstack([y, -np.ones(n_extra)]).astype(np.float32)
    return X_new, y_new

### Phase A: Plot data and decision boundary

Boundary is the line $w^\top x + b = 0$. Below: data and boundary at the end of Phase A.

In [ ]:
def plot_boundary(X, y, w, b, title='Decision boundary',
                  n_phase_a=None, xlim=XLIM, ylim=YLIM):
    """Plot data points and decision boundary.

    Parameters
    ----------
    X, y        : data (all points in current phase)
    w, b        : perceptron weights
    title       : plot title
    n_phase_a   : if not None, the first n_phase_a points with y==-1 are the
                  original Class B cluster (μ2, squares); the remainder are the
                  new μ3 cluster (triangles).
    xlim, ylim  : fixed axis limits
    """
    fig, ax = plt.subplots(figsize=(5, 5))

    # Class A — blue circles
    ax.scatter(X[y == 1, 0], X[y == 1, 1],
               c='tab:blue', marker='o', alpha=0.7, label='Class A (μ1)')

    # Class B — split by original / new cluster when n_phase_a is given
    b_mask = (y == -1)
    b_idx  = np.where(b_mask)[0]
    if n_phase_a is not None and len(b_idx) > n_phase_a:
        orig_idx = b_idx[:n_phase_a]
        new_idx  = b_idx[n_phase_a:]
        ax.scatter(X[orig_idx, 0], X[orig_idx, 1],
                   c='tab:orange', marker='s', alpha=0.7, label='Class B (μ2)')
        ax.scatter(X[new_idx, 0],  X[new_idx, 1],
                   c='tab:orange', marker='^', alpha=0.7, label='Class B new cluster (μ3)')
    else:
        ax.scatter(X[b_mask, 0], X[b_mask, 1],
                   c='tab:orange', marker='s', alpha=0.7, label='Class B (μ2)')

    # Centroids
    for mu, label in [(MU1, 'μ1'), (MU2, 'μ2'), (MU3, 'μ3')]:
        ax.scatter(*mu, c='k', marker='x', s=120, zorder=5, linewidths=2)
        ax.annotate(label, mu, textcoords='offset points', xytext=(6, 4), fontsize=10)

    # Decision boundary line
    if abs(w[1]) > 1e-8:
        x1_line = np.linspace(xlim[0], xlim[1], 400)
        x2_line = -(w[0] * x1_line + b) / w[1]
        ax.plot(x1_line, x2_line, 'k-', lw=2, label='Boundary')

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.legend(loc='upper left', fontsize=8)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

# Pre-generate the extended dataset so Phase B reuses the same samples
X_ext, y_ext = add_third_cluster(X, y)
n_b_phase_a   = int(np.sum(y == -1))   # number of original Class B points

plot_boundary(X, y, w, b,
              title='Phase A: boundary after training (2 clusters)')

### Phase B: New data arrives (do not reinitialise)

We add a **third Gaussian cluster** belonging to **Class B** (label −1), placed so that the optimal boundary changes (e.g. top-left). We **continue training from the current** `w`, `b`—do **not** reinitialise. The boundary will shift to include the new cluster on the correct side. Run the cells below and compare the plot to Phase A.

#### Pause & Predict

Before running Phase B, answer briefly (or discuss):

1. **What do you expect the decision boundary to do** when we add the third cluster and continue training?

2. **What might happen to accuracy on the original two clusters** after the boundary shifts to include the new cluster?

In [ ]:
# X_ext, y_ext and n_b_phase_a were computed in the Phase A plot cell above.

In [ ]:
# Continue training from current w, b (perceptron with warm start)
def perceptron_fit_continue(X, y, w_start, b_start, lr=0.1, max_epochs=150):
    n, d = X.shape
    w, b = w_start.copy(), float(b_start)
    for epoch in range(max_epochs):
        mistakes = 0
        for i in range(n):
            z = np.dot(w, X[i]) + b
            pred = 1 if z >= 0 else -1
            if pred != y[i]:
                mistakes += 1
                w = w + lr * y[i] * X[i]
                b = b + lr * y[i]
        if mistakes == 0:
            print(f"Converged at epoch {epoch + 1}")
            break
    return w, b

w2, b2 = perceptron_fit_continue(X_ext, y_ext, w, b)
plot_boundary(X_ext, y_ext, w2, b2,
              title='Phase B: boundary after continuing training (new Class B cluster added)',
              n_phase_a=n_b_phase_a)

Continuing training from the current weights adapts the model to the new empirical distribution: the boundary **shifts and rotates** so that the μ3 cluster (Class B, top-left) falls on the correct side. Because the boundary has moved, some points from the original two clusters may now be misclassified—the perceptron is no longer optimised for the earlier data alone. This accuracy trade-off motivates **mini-batching over the full dataset** and later techniques (e.g. replay, regularisation) for maintaining stability when the data distribution or task objectives change.

#### Pause & Reflect

Before adding the third cluster, the boundary separated the two original clusters (Class A and Class B). After continuing training, it had to move to include the new cluster on the correct side. How would the boundary have looked if we had trained from scratch on all three clusters from the beginning? Would it be the same?

### TODO 2

Change the third cluster's `centroid` to `(0, 0)` so it sits between the two original clusters. Re-run the data generation and continued training. Does the perceptron still converge? Why or why not?

### TODO 3 (perceptron section)

Switch training from **online** updates (one update per sample) to **mini-batch** updates: e.g. accumulate the perceptron update over $k$ samples, then apply once per mini-batch. Compare the final boundary (and its stability across runs) with the online version. Hint: you can sum the $\eta y_i x_i$ and $\eta y_i$ contributions for misclassified points in the batch, then add once to $w$ and $b$.

---
## 3. Practical 2: XOR failure (visual)

The four XOR points in 2D: class 0 at $(0,0)$ and $(1,1)$; class 1 at $(0,1)$ and $(1,0)$. No single line can separate them. We plot the points and a sample "best" linear boundary to show it fails.

**Intuition:** A hidden layer can be thought of as first *transforming* the input space—bending and stretching it—so that the two XOR classes become linearly separable in the new representation before the final output layer sees them.

In [ ]:
X_xor = np.array([[0., 0.], [1., 1.], [0., 1.], [1., 0.]], dtype=np.float32)
y_xor = np.array([-1., -1., 1., 1.], dtype=np.float32)  # class -1: (0,0),(1,1); class +1: (0,1),(1,0)

plt.figure(figsize=(5, 5))
plt.scatter(X_xor[y_xor == 1, 0], X_xor[y_xor == 1, 1], c='C0', s=120, label='Class +1', marker='s')
plt.scatter(X_xor[y_xor == -1, 0], X_xor[y_xor == -1, 1], c='C1', s=120, label='Class -1', marker='o')
# Draw a diagonal line (any single line misclassifies at least one point)
xx = np.linspace(-0.2, 1.2, 2)
plt.plot(xx, xx, 'k--', lw=1.5, label='Example line (fails)')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.legend()
plt.title('XOR: no single line can separate the classes')
plt.gca().set_aspect('equal')
plt.tight_layout()
plt.show()

#### Pause & Reflect

- The perceptron is guaranteed to converge only when data are linearly separable. What do you expect `perceptron_fit` to report when run on `X_xor`—convergence or reaching `max_epochs`?
- With only four points, could any placement of a single straight line correctly classify all of them as XOR labels? Sketch or argue why not.

### TODO 3

Train the same NumPy perceptron on `X_xor`, `y_xor` with `max_epochs=500`. Does it converge? Print the number of mistakes per epoch to see if it gets stuck in a loop.

---
## 4. Practical 3: Tiny MLP solves XOR

We use PyTorch to define a small MLP (e.g. 2 → 2 → 1 with non-linear activation). With gradient descent and a differentiable loss, the network can learn a decision boundary that separates XOR.

In [ ]:
import torch.nn as nn

class TinyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 2)
        self.fc2 = nn.Linear(2, 1)

    def forward(self, x):
        x = torch.tanh(self.fc1(x))  # non-linearity essential
        return self.fc2(x).squeeze(-1)

torch.manual_seed(0)  # fixed seed → stable convergence every run
model_xor = TinyMLP().to(device)
opt = torch.optim.SGD(model_xor.parameters(), lr=0.5)
X_t = torch.from_numpy(X_xor).to(device)
y_t = torch.from_numpy(y_xor).to(device).float()  # 1.0 and -1.0

In [ ]:
for epoch in range(2000):
    model_xor.train()
    opt.zero_grad()
    logits = model_xor(X_t)
    # Binary classification: use BCE with logits; targets in {0,1} for BCE
    targets = (y_t + 1) / 2  # -1,1 -> 0,1
    loss = nn.functional.binary_cross_entropy_with_logits(logits, targets)
    loss.backward()
    opt.step()
    if (epoch + 1) % 500 == 0:
        pred = (torch.sigmoid(logits) > 0.5).float() * 2 - 1
        acc = (pred == y_t).float().mean().item()
        print(f"Epoch {epoch+1:4d}, loss={loss.item():.4f}, acc={acc:.2f}")

### TODO: Ablation — skip the optimizer step

In the XOR training loop above, comment out `opt.step()` (add `#` before it) and re-run for 2000 epochs. Record the final loss. Then restore `opt.step()` and re-run again. By how many orders of magnitude does the final loss differ between the two runs? Why does the loss stop decreasing without the optimizer step even though `loss.backward()` still runs?

In [ ]:
model_xor.eval()
with torch.no_grad():
    logits_eval = model_xor(X_t)
    probs_eval  = torch.sigmoid(logits_eval)
    preds_eval  = (probs_eval > 0.5).float() * 2 - 1  # back to {-1, +1}
    acc_final   = (preds_eval == y_t).float().mean().item()

print(f"{'Point':>12}  {'True':>5}  {'Logit':>7}  {'P(+1)':>7}  {'Pred':>5}")
print("-" * 46)
for xi, yt, lg, pb, pd in zip(X_xor, y_xor,
                                logits_eval.cpu().numpy(),
                                probs_eval.cpu().numpy(),
                                preds_eval.cpu().numpy()):
    print(f"({xi[0]:.0f}, {xi[1]:.0f})       {yt:>+.0f}   {lg:>+7.3f}  {pb:>7.4f}  {pd:>+.0f}")
print("-" * 46)
print(f"Accuracy: {acc_final:.2f}  ({'✓ all correct' if acc_final == 1.0 else '✗ not all correct'})")

In [ ]:
# Decision boundary in 2D: grid and model prediction
def plot_mlp_boundary(model, X_data, y_data, device, title='MLP decision boundary'):
    model.eval()
    x1 = np.linspace(-0.5, 1.5, 80)
    x2 = np.linspace(-0.5, 1.5, 80)
    X1, X2 = np.meshgrid(x1, x2)
    grid = np.stack([X1.ravel(), X2.ravel()], axis=1).astype(np.float32)
    with torch.no_grad():
        logits = model(torch.from_numpy(grid).to(device))
        probs = torch.sigmoid(logits).cpu().numpy()
    probs = probs.reshape(X1.shape)
    plt.figure(figsize=(5, 5))
    plt.contourf(X1, X2, probs, levels=20, cmap='RdBu_r', alpha=0.6)
    plt.colorbar(label='P(class +1)')

    # Confidence contours: thin grey lines at 0.25 and 0.75
    cs_conf = plt.contour(X1, X2, probs, levels=[0.25, 0.75],
                          colors=['#555555'], linewidths=1.0, linestyles='--')
    plt.clabel(cs_conf, fmt={0.25: 'p = 0.25', 0.75: 'p = 0.75'},
               inline=True, fontsize=8)

    # Decision boundary: thick black line at p = 0.5
    cs_bd = plt.contour(X1, X2, probs, levels=[0.5],
                        colors='k', linewidths=2.5)
    plt.clabel(cs_bd, fmt={0.5: 'p = 0.5'}, inline=True, fontsize=9)

    plt.scatter(X_data[y_data == 1, 0], X_data[y_data == 1, 1],
                c='C0', s=100, edgecolors='k', label='Class +1', zorder=3)
    plt.scatter(X_data[y_data == -1, 0], X_data[y_data == -1, 1],
                c='C1', s=100, edgecolors='k', label='Class -1', zorder=3)

    # Invisible proxy artists for legend notes
    from matplotlib.lines import Line2D
    handles_extra = [
        Line2D([0], [0], color='k',        lw=2.5,              label='p = 0.5  (decision threshold)'),
        Line2D([0], [0], color='#555555',   lw=1.0, ls='--',    label='p = 0.25 / 0.75  (confidence regions)'),
    ]
    handles, labels = plt.gca().get_legend_handles_labels()
    plt.legend(handles=handles + handles_extra, fontsize=8, loc='upper right')

    plt.xlabel('$x_1$')
    plt.ylabel('$x_2$')
    plt.title(title)
    plt.gca().set_aspect('equal')
    plt.tight_layout()
    plt.show()

plot_mlp_boundary(model_xor, X_xor, y_xor, device,
                  'Tiny MLP on XOR — decision boundary and confidence contours')

#### Pause & Reflect

- Look at the curved decision boundary the MLP learned. How does it qualitatively differ from the straight-line boundary a single perceptron can produce?
- The hidden layer maps each input $(x_1, x_2)$ to a 2-D intermediate representation before the output neuron. In that new space, why are the two XOR classes now linearly separable?

### TODO 4

Replace `tanh` with a linear layer (remove the activation between the two layers) and re-train. Plot the boundary again. Why does the boundary stay a single line?

---
## 5. Practical 4: Minimal MNIST training loop

We run 1–2 epochs of a standard training loop on MNIST with a small MLP. This reinforces: batch, epoch, forward, loss, backward, step.

### Load MNIST

In [ ]:
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
print(f"Train samples: {len(train_dataset)}, batches per epoch: {len(train_loader)}")

### TODO 5

Set `batch_size=32` and re-run the DataLoader cell. How many batches per epoch do you get? Confirm it matches 60000 / 32.

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_size=784, hidden_size=128, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

model_mnist = MLP(784, 128, 10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_mnist.parameters(), lr=1e-3)

In [ ]:
num_epochs = 2
for epoch in range(num_epochs):
    model_mnist.train()
    running_loss = 0.0
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        logits = model_mnist(data)
        loss = criterion(logits, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{num_epochs}, train loss: {avg_loss:.4f}")

#### Pause & Reflect

- The running loss is printed every 100 batches. Does it trend downward within an epoch, or does it fluctuate? What is the source of those fluctuations?
- After epoch 2 the average loss is lower than after epoch 1. What would you expect if you ran 10 epochs—would the loss keep dropping at the same rate, speed up, or slow down?

### TODO 6

Add a simple accuracy computation over the training set after each epoch (e.g. run one full pass without updating, count correct predictions). Print epoch and accuracy.

### TODO: Accuracy breakdown by class

After computing overall training accuracy (as in TODO 6), extend the code to also print the **per-digit accuracy** (one line per digit 0–9). For each digit $d$, count how many examples of that digit the model predicted correctly and divide by the total number of examples of digit $d$ in the training set. Which digit has the lowest accuracy? Is that surprising given how the digit looks?

#### Pause & Reflect

In the MNIST loop we use `CrossEntropyLoss` and logits (no softmax in the forward). Why is it correct to pass logits to `CrossEntropyLoss`? What would happen if we applied softmax in the model and then used a different loss?

---
## Week 2 Takeaways

- **Single neuron = one hyperplane.** $z = w^\top x + b$ defines a linear decision boundary; the sign of $z$ tells you which side a point is on, and the magnitude is a (rough) confidence measure.
- **Training loop mechanics.** Every parameter update in deep learning follows the same four-step cycle: forward pass → loss → backward pass → optimizer step. Internalising this cycle is the foundation for reading any modern training code.
- **Why nonlinearity matters.** A stack of purely linear layers collapses to a single linear map. Non-linear activations (tanh, ReLU, …) let hidden layers carve out curved regions, enabling networks to solve problems—like XOR—that are provably impossible for a single perceptron.
- **Representation learning starts here.** Hidden layers don't merely classify; they learn a new *representation* of the input. The tiny MLP that solves XOR works by mapping the four points into a space where they become linearly separable. This idea, scaled to thousands of layers, is the engine behind modern deep learning.

**Looking ahead to Week 3:** We will examine how the choice of optimizer (SGD, Adam, …) and learning-rate schedule affects convergence stability, and introduce regularisation techniques—weight decay, dropout—that prevent a model from memorising the training set at the expense of generalisation.

---
## 6. Wrap-up: Debate prompts

Discuss in small groups or write short answers:

1. **Perceptron vs gradient-based learning:** The perceptron update rule does not minimise a single global loss; it only reacts to mistakes. In what situations might such a rule be preferable to gradient descent on a smooth loss? When is it strictly worse?

2. **Batch size:** "Larger batches give more stable gradients but fewer parameter updates per epoch." How would you trade off batch size for a fixed compute budget (e.g. one epoch)? What could go wrong with batch size 1 vs batch size 60000?

3. **XOR and depth:** We solved XOR with one hidden layer (2→2→1). Could we have used a deeper network with one neuron per layer? What is the minimal architecture (number of layers and units) you need to represent the XOR decision boundary, and does that match what you implemented?